In [31]:
import torch
import platform
import subprocess
import os

print("=" * 60)
print("SYSTEM INFORMATION")
print("=" * 60)

# Basic system info
print(f"Hostname: {platform.node()}")
print(f"OS: {platform.system()} {platform.release()}")
print(f"Python: {platform.python_version()}")
print(f"Python Executable: {os.sys.executable}")

print("\n" + "=" * 60)
print("CPU INFORMATION")
print("=" * 60)

# CPU info
try:
    cpu_info = subprocess.check_output("lscpu | grep 'Model name'", shell=True).decode()
    print(cpu_info.strip())
    cpu_count = os.cpu_count()
    print(f"CPU Cores Available: {cpu_count}")
except:
    print(f"CPU Cores: {os.cpu_count()}")

print("\n" + "=" * 60)
print("MEMORY INFORMATION")
print("=" * 60)

# Memory info
try:
    mem_info = subprocess.check_output("free -h | grep Mem", shell=True).decode()
    parts = mem_info.split()
    print(f"Total RAM: {parts[1]}")
    print(f"Used RAM: {parts[2]}")
    print(f"Available RAM: {parts[6]}")
except:
    print("Memory info unavailable")

print("\n" + "=" * 60)
print("GPU INFORMATION")
print("=" * 60)

# PyTorch CUDA info
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"cuDNN Version: {torch.backends.cudnn.version()}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    
    # Detailed info for each GPU
    for i in range(torch.cuda.device_count()):
        print(f"\n--- GPU {i} ---")
        print(f"Name: {torch.cuda.get_device_name(i)}")
        props = torch.cuda.get_device_properties(i)
        print(f"Compute Capability: {props.major}.{props.minor}")
        print(f"Total Memory: {props.total_memory / 1024**3:.2f} GB")
        print(f"Memory Allocated: {torch.cuda.memory_allocated(i) / 1024**3:.2f} GB")
        print(f"Memory Reserved: {torch.cuda.memory_reserved(i) / 1024**3:.2f} GB")
        print(f"Multi-Processors: {props.multi_processor_count}")
else:
    print("No CUDA GPUs detected")

print("\n" + "=" * 60)
print("STORAGE INFORMATION")
print("=" * 60)

# Disk space
try:
    disk_info = subprocess.check_output(f"df -h {os.path.expanduser('~')}", shell=True).decode()
    lines = disk_info.strip().split('\n')
    if len(lines) > 1:
        print(lines[0])  # Header
        print(lines[1])  # Home directory info
except:
    print("Disk info unavailable")

print("=" * 60)

SYSTEM INFORMATION
Hostname: nodegpu113.hpc.fau.edu
OS: Linux 4.18.0-553.58.1.el8_10.x86_64
Python: 3.13.11
Python Executable: /mnt/beegfs/home/yyu2024/my_pytorch_env/bin/python

CPU INFORMATION
Model name:          Intel(R) Xeon(R) Gold 6130 CPU @ 2.10GHz
CPU Cores Available: 64

MEMORY INFORMATION
Total RAM: 187Gi
Used RAM: 13Gi
Available RAM: 161Gi

GPU INFORMATION
PyTorch Version: 2.6.0+cu124
CUDA Available: True
CUDA Version: 12.4
cuDNN Version: 90100
Number of GPUs: 4

--- GPU 0 ---
Name: Tesla V100-SXM2-32GB
Compute Capability: 7.0
Total Memory: 31.73 GB
Memory Allocated: 0.00 GB
Memory Reserved: 0.03 GB
Multi-Processors: 80

--- GPU 1 ---
Name: Tesla V100-SXM2-32GB
Compute Capability: 7.0
Total Memory: 31.73 GB
Memory Allocated: 0.00 GB
Memory Reserved: 0.00 GB
Multi-Processors: 80

--- GPU 2 ---
Name: Tesla V100-SXM2-32GB
Compute Capability: 7.0
Total Memory: 31.73 GB
Memory Allocated: 0.00 GB
Memory Reserved: 0.00 GB
Multi-Processors: 80

--- GPU 3 ---
Name: Tesla V100-SXM2-3

In [32]:
import sys
print(sys.executable)

!{sys.executable} -m pip install numpy mne scipy plotly pandas scikit-learn pytorch-model-summary wandb


/mnt/beegfs/home/yyu2024/my_pytorch_env/bin/python


In [ ]:
import numpy as np
import scipy.io, scipy.interpolate
import pathlib
import matplotlib.pyplot as plt
import torch
import pytorch_lightning as pl
from pytorch_model_summary import summary
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import scipy.ndimage
import plotly.tools as tls
from pytorch_lightning.callbacks import Callback
from pytorch_lightning import Trainer
import wandb
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint

L_FREQ, H_FREQ = 40, 300 # Lower and upper filtration bounds
CHANNELS_NUM = 62        # Number of channels in ECoG data
WAVELET_NUM = 40         # Number of wavelets in the indicated frequency range, with which the convolution is performed
DOWNSAMPLE_FS = 100      # Desired sampling rate
time_delay_secs = 0.2    # Time delay hyperparameter

current_fs = DOWNSAMPLE_FS

# =============================================================================
# MODEL SELECTION
# =============================================================================
# Set to 'fingerflex' for original FingerFlex U-Net model
# Set to 'bc4d4' for BC4D4 CNN+DNN model (Jangir et al. 2025)
MODEL_MODE = 'bc4d4'  # <-- CHANGE THIS TO SWITCH MODELS

# =============================================================================
# BC4D4 SETTINGS
# =============================================================================
# IMPORTANT: Paper trains 5 SEPARATE models, one per finger!
# Set BC4D4_FINGER_IDX to 0-4 to train a single finger model (recommended)
# Set to None to train all 5 fingers at once (not paper methodology)
BC4D4_ACTIVATION = 'softsign'  # 'tanh' or 'softsign' (softsign gives best results)
BC4D4_FINGER_IDX = 0  # <-- TRAIN ONE FINGER: 0=Thumb, 1=Index, 2=Middle, 3=Ring, 4=Little

FINGER_NAMES = ['Thumb', 'Index', 'Middle', 'Ring', 'Little']

TYPE = "train"  # Script modes: "train" and "test"
model_to_test = f"{pathlib.Path().resolve()}/checkpoints/model-epoch=16-corr_mean_val=0.6790410876274109.ckpt"

print(f"Model mode: {MODEL_MODE}")
if MODEL_MODE == 'bc4d4':
    if BC4D4_FINGER_IDX is not None:
        print(f"BC4D4 finger: {FINGER_NAMES[BC4D4_FINGER_IDX]} (index {BC4D4_FINGER_IDX})")
        print("NOTE: Paper methodology trains 5 separate models. Run this 5 times with BC4D4_FINGER_IDX=0,1,2,3,4")
    else:
        print("BC4D4: Training all 5 fingers at once (NOT paper methodology)")
print(f"Run type: {TYPE}")

In [ ]:
# =============================================================================
# DATASET CLASSES
# =============================================================================

class EcogFingerflexDataset(Dataset):
    """
    Dataset for FingerFlex model (wavelet spectrograms)
    ECoG shape: (channels, wavelets, time)
    """
    def __init__(self, path_to_ecog_data: str,
                 path_to_fingerflex_data: str, sample_len: int, train = False):
        """
        paths should point to .npy files
        """
        self.ecog_data, self.fingerflex_data = np.load(path_to_ecog_data).astype('float32'),\
                                            np.load(path_to_fingerflex_data).astype('float32')
        
        self.duration = self.ecog_data.shape[2]
        self.sample_len = sample_len                                 # sample size
        self.stride = 1                                              # stride between samples
        self.ds_len = (self.duration-self.sample_len) // self.stride
        self.train = train
        
        print("Duration: ", self.duration, "Ds_len:", self.ds_len)
    def __len__(self):
        return self.ds_len
    
    def __getitem__(self, index):

        sample_start = index*self.stride
        sample_end = sample_start+self.sample_len

        ecog_sample = self.ecog_data[...,sample_start:sample_end] # x
        
        fingerflex_sample = self.fingerflex_data[...,sample_start:sample_end] # y
        
        return ecog_sample, fingerflex_sample


class BC4D4Dataset(Dataset):
    """
    Dataset for BC4D4 model (raw ECoG signals)
    ECoG shape: (time, electrodes, 1)
    Finger shape: (time, 5)
    
    BC4D4 predicts ONE finger at a time (per-finger models)
    """
    def __init__(self, path_to_ecog_data: str,
                 path_to_fingerflex_data: str, 
                 finger_idx: int = None,
                 train: bool = False):
        """
        Parameters
        ----------
        path_to_ecog_data : str
            Path to ECoG .npy file (time, electrodes, 1)
        path_to_fingerflex_data : str
            Path to finger .npy file (time, 5)
        finger_idx : int
            Which finger to predict (0-4). If None, predict all 5.
        """
        self.ecog_data = np.load(path_to_ecog_data).astype('float32')
        self.fingerflex_data = np.load(path_to_fingerflex_data).astype('float32')
        self.finger_idx = finger_idx
        self.train = train
        
        # BC4D4 data shape: (time, electrodes, 1)
        self.n_samples = self.ecog_data.shape[0]
        
        print(f"BC4D4 Dataset - Samples: {self.n_samples}, "
              f"Electrodes: {self.ecog_data.shape[1]}, "
              f"Finger idx: {finger_idx}")
        
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, index):
        ecog_sample = self.ecog_data[index]  # (electrodes, 1)
        
        if self.finger_idx is not None:
            # Single finger prediction
            finger_sample = self.fingerflex_data[index, self.finger_idx:self.finger_idx+1]  # (1,)
        else:
            # All fingers
            finger_sample = self.fingerflex_data[index]  # (5,)
        
        return ecog_sample, finger_sample


class EcogFingerflexDatamodule(pl.LightningDataModule):
    """
    A class that encapsulates different datasets (for training and validation) and their dataloaders
    Supports both FingerFlex and BC4D4 data formats
    """
    def __init__(self, sample_len: int, data_dir = f"{pathlib.Path().resolve()}/data",
                    batch_size=128, add_name="", model_mode='fingerflex', finger_idx=None):
        super().__init__()
        self.data_dir = data_dir     # Path to data folder
        self.sample_len = sample_len # Sample size (used for FingerFlex)
        self.batch_size = batch_size # Dataloader batch size
        self.add_name = add_name     # dataset name suffix
        self.model_mode = model_mode # 'fingerflex' or 'bc4d4'
        self.finger_idx = finger_idx # For BC4D4: which finger (0-4) or None for all
        
    def setup(self, stage = None):
        if self.model_mode == 'fingerflex':
            # FingerFlex datasets
            if stage is None or stage == "fit":
                self.train = EcogFingerflexDataset(f"{self.data_dir}/train/ecog_data{self.add_name}.npy",
                                                  f"{self.data_dir}/train/fingerflex_data{self.add_name}.npy",
                                                  self.sample_len, train = True)
                
                self.val = EcogFingerflexDataset(f"{self.data_dir}/val/ecog_data{self.add_name}.npy",
                                                  f"{self.data_dir}/val/fingerflex_data{self.add_name}.npy",
                                                  self.sample_len)
            
            if stage is None or stage == "test":
                self.test = EcogFingerflexDataset(f"{self.data_dir}/test/ecog_data{self.add_name}.npy",
                                                  f"{self.data_dir}/test/fingerflex_data{self.add_name}.npy",
                                                  self.sample_len)
        
        elif self.model_mode == 'bc4d4':
            # BC4D4 datasets
            if stage is None or stage == "fit":
                self.train = BC4D4Dataset(f"{self.data_dir}/train/ecog_data{self.add_name}.npy",
                                          f"{self.data_dir}/train/fingerflex_data{self.add_name}.npy",
                                          finger_idx=self.finger_idx, train=True)
                
                self.val = BC4D4Dataset(f"{self.data_dir}/val/ecog_data{self.add_name}.npy",
                                        f"{self.data_dir}/val/fingerflex_data{self.add_name}.npy",
                                        finger_idx=self.finger_idx)
            
            if stage is None or stage == "test":
                self.test = BC4D4Dataset(f"{self.data_dir}/test/ecog_data{self.add_name}.npy",
                                         f"{self.data_dir}/test/fingerflex_data{self.add_name}.npy",
                                         finger_idx=self.finger_idx)
    
    def train_dataloader(self):
        return DataLoader(self.train, batch_size=self.batch_size, num_workers=4, shuffle=True)
    
    def val_dataloader(self):
        return DataLoader(self.val, batch_size=self.batch_size)
    
    def test_dataloader(self):
        return DataLoader(self.test, batch_size=self.batch_size)

In [ ]:
def correlation_metric(x, y):
    """
     Cosine distance calculation metric
    """
    cos_metric = nn.CosineSimilarity(dim=-1, eps=1e-08)

    cos_sim = torch.mean(cos_metric(x, y))

    return cos_sim

def corr_metric(x, y):
    """
    Pearson correlation calculation metric between univariate vectors
    """
    assert x.shape == y.shape  
    r = np.corrcoef(x, y)[0, 1]
    return r


class BaseEcogFingerflexModel(pl.LightningModule):
    """
    Lightning wrapper for FingerFlex model
    Uses cosine similarity + MSE loss
    """
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.lr = 8.42e-5
        
    def training_step(self, batch, batch_idx):
        x, y = batch
        
        y_hat = self.model(x)
        
        loss = F.mse_loss(y_hat, y)
        corr = correlation_metric(y_hat, y)

        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        self.log(f"cosine_dst_train", corr, on_step=False, on_epoch=True, prog_bar=True, logger=True)

        return 0.5*loss + 0.5*(1. - corr)
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self.model(x)
        loss = F.mse_loss(y_hat, y)
        
        corr = correlation_metric(y_hat, y)

        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        self.log("cosine_dst_val", corr, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        
        return y_hat
    
    def test_step(self, batch, batch_idx):
        x, y = batch
        
        y_hat = self.model(x)
        
        loss = F.mse_loss(y_hat, y)
        self.log("test_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=1e-6)
        return optimizer


class BC4D4LightningModel(pl.LightningModule):
    """
    Lightning wrapper for BC4D4 model
    Uses MSE loss (regression task)
    """
    def __init__(self, model, lr=1e-3):
        super().__init__()
        self.model = model
        self.lr = lr
        
    def training_step(self, batch, batch_idx):
        x, y = batch
        
        y_hat = self.model(x)
        
        loss = F.mse_loss(y_hat, y)
        
        # Calculate Pearson correlation for monitoring
        with torch.no_grad():
            y_np = y.cpu().numpy().flatten()
            y_hat_np = y_hat.cpu().numpy().flatten()
            corr = np.corrcoef(y_np, y_hat_np)[0, 1] if len(y_np) > 1 else 0

        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        self.log("train_corr", corr, on_step=False, on_epoch=True, prog_bar=True, logger=True)

        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self.model(x)
        loss = F.mse_loss(y_hat, y)
        
        # Calculate Pearson correlation
        with torch.no_grad():
            y_np = y.cpu().numpy().flatten()
            y_hat_np = y_hat.cpu().numpy().flatten()
            corr = np.corrcoef(y_np, y_hat_np)[0, 1] if len(y_np) > 1 else 0

        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        self.log("val_corr", corr, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        
        return y_hat
    
    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self.model(x)
        loss = F.mse_loss(y_hat, y)
        
        with torch.no_grad():
            y_np = y.cpu().numpy().flatten()
            y_hat_np = y_hat.cpu().numpy().flatten()
            corr = np.corrcoef(y_np, y_hat_np)[0, 1] if len(y_np) > 1 else 0
        
        self.log("test_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        self.log("test_corr", corr, on_step=False, on_epoch=True, prog_bar=True, logger=True)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=0)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=10, verbose=True
        )
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss',
            }
        }


print("Lightning wrappers loaded: BaseEcogFingerflexModel, BC4D4LightningModel")

In [ ]:
"""
Model architectures: FingerFlex (U-Net) and BC4D4 (CNN+DNN)
"""

# =============================================================================
# FINGERFLEX MODEL (Original U-Net Architecture)
# =============================================================================

class ConvBlock(nn.Module):
    """
    Convolution block:
        - 1d conv
        - layer norm by embedding axis
        - activation
        - dropout
        - Max pooling
    """
    def __init__(self, in_channels, out_channels, kernel_size, 
                 stride=1, dilation=1, p_conv_drop=0.1):
        super(ConvBlock, self).__init__()
        
        # use it instead stride. 
        
        self.conv1d = nn.Conv1d(in_channels, out_channels, 
                                kernel_size=kernel_size, 
                                bias=False, 
                                padding='same')
        
        
        self.norm = nn.LayerNorm(out_channels)
        self.activation = nn.GELU()
        self.drop = nn.Dropout(p=p_conv_drop)

        self.downsample = nn.MaxPool1d(kernel_size=stride, stride=stride)

        self.stride = stride
        self.in_channels = in_channels
        self.out_channels = out_channels
        
        
    def forward(self, x):
        
        x = self.conv1d(x)
        
        # norm by last axis.
        x = torch.transpose(x, -2, -1) 
        x = self.norm(x)
        x = torch.transpose(x, -2, -1)
        
        x = self.activation(x)
        
        x = self.drop(x)
        
        x = self.downsample(x)

        return x

    
    
    
    
class UpConvBlock(nn.Module):
    """
    Decoder convolution block
    """
    def __init__(self, scale, **args):
        super(UpConvBlock, self).__init__()
        self.conv_block = ConvBlock(**args)
        self.upsample = nn.Upsample(scale_factor=scale, mode='linear', align_corners=False)

            
    def forward(self, x):
        
        x = self.conv_block(x)
        x = self.upsample(x)
        return x    
    
    


    
    
class AutoEncoder1D(nn.Module):
    """
    This is the final Encoder-Decoder model with skip connections (FingerFlex)
    """
    def __init__(self,
                 n_electrodes=30,   # Number of channels
                 n_freqs = 16,      # Number of wavelets
                 n_channels_out=21, # Number of fingers
                 channels = [8, 16, 32, 32],  # Number of features on each encoder layer
                 kernel_sizes=[3, 3, 3],
                 strides=[4, 4, 4],
                 dilation=[1, 1, 1]
                 ):
        
        super(AutoEncoder1D, self).__init__()
        

        self.n_electrodes = n_electrodes
        self.n_freqs = n_freqs
        self.n_inp_features = n_freqs*n_electrodes
        self.n_channels_out = n_channels_out
        
        self.model_depth = len(channels)-1
        self.spatial_reduce = ConvBlock(self.n_inp_features, channels[0], kernel_size=3) # Dimensionality reduction
        
        # Encoder part
        self.downsample_blocks = nn.ModuleList([ConvBlock(channels[i], 
                                                        channels[i+1], 
                                                        kernel_sizes[i],
                                                        stride=strides[i], 
                                                        dilation=dilation[i]) for i in range(self.model_depth)])
        

        channels = [ch for ch in channels[:-1]] + channels[-1:] # channels

        # Decoder part
        self.upsample_blocks = nn.ModuleList([UpConvBlock(scale=strides[i],
                                                          in_channels=channels[i+1] if i == self.model_depth-1 else channels[i+1]*2 ,
                                                          out_channels=channels[i],
                                                          kernel_size=kernel_sizes[i]) for i in range(self.model_depth-1, -1, -1)])
        
        
        self.conv1x1_one = nn.Conv1d(channels[0]*2, self.n_channels_out, kernel_size=1, padding='same') # final 1x1 conv
      
    def forward(self, x):

        batch, elec, n_freq, time = x.shape
        x = x.reshape(batch, -1, time)  # flatten the input
        x = self.spatial_reduce(x)
        
        skip_connection = []
        
        for i in range(self.model_depth):
            skip_connection.append(x)
            x = self.downsample_blocks[i](x)

        
        for i in range(self.model_depth):
            x = self.upsample_blocks[i](x)
            x = torch.cat((x, skip_connection[-1 - i]), # skip connections
                         dim=1)
        
        x = self.conv1x1_one(x)

        return x


# =============================================================================
# BC4D4 MODEL (Jangir et al. 2025)
# =============================================================================
# Architecture from paper Table 3:
# - CNN Block: 3 Conv1D layers with ReLU (no pooling)
# - DNN Block: 6 Dense layers with Tanh/Softsign activation
# - Output: 1 (single finger) or 5 (all fingers)

class Softsign(nn.Module):
    """
    Softsign activation function: f(x) = x / (1 + |x|)
    
    Properties:
    - Range: [-1, +1]
    - Grows polynomially (gentler than Tanh)
    - Avoids vanishing gradient better than Tanh
    - Paper shows ~5% better correlation than Tanh
    """
    def forward(self, x):
        return x / (1 + torch.abs(x))


class BC4D4(nn.Module):
    """
    BC4D4 CNN+DNN model for BCI finger movement regression.
    
    Architecture from paper Table 3:
    - CNN Block: Conv1D(1->64->128->256) with ReLU, kernel=3, NO pooling
    - DNN Block: Dense(1024->512->256->128->64->output) with Tanh/Softsign
    
    IMPORTANT: This model predicts ONE finger at a time by default.
    Set n_outputs=5 to predict all fingers.
    
    Parameters
    ----------
    num_features : int
        Number of ECoG electrodes (62 for Subject 1)
    activation : str
        'tanh' or 'softsign' (softsign recommended)
    dropout_rate : float
        Dropout rate (paper uses 0.1)
    n_outputs : int
        Number of output values (1 for single finger, 5 for all)
    """
    def __init__(self, num_features: int, activation: str = 'softsign',
                 dropout_rate: float = 0.1, n_outputs: int = 1):
        super(BC4D4, self).__init__()
        
        self.num_features = num_features
        self.activation_name = activation
        self.n_outputs = n_outputs
        
        # CNN Block (ReLU activation)
        # NO padding, NO pooling (paper states pooling causes info loss)
        self.conv1 = nn.Conv1d(1, 64, kernel_size=3, stride=1, padding=0)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, stride=1, padding=0)
        self.conv3 = nn.Conv1d(128, 256, kernel_size=3, stride=1, padding=0)
        
        # Flatten size after CNN: (num_features - 6) * 256
        self.flatten_size = (num_features - 6) * 256
        
        # Select DNN activation
        if activation == 'tanh':
            self.dense_activation = nn.Tanh()
        elif activation == 'softsign':
            self.dense_activation = Softsign()
        else:
            raise ValueError(f"Unknown activation: {activation}")
        
        # DNN Block
        self.fc1 = nn.Linear(self.flatten_size, 1024)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 256)
        self.fc4 = nn.Linear(256, 128)
        self.fc5 = nn.Linear(128, 64)
        self.fc6 = nn.Linear(64, n_outputs)
    
    def forward(self, x):
        """
        Forward pass.
        
        Input shape: (batch, num_features, 1)
        Output shape: (batch, n_outputs)
        """
        # Permute to (batch, 1, num_features) for Conv1d
        x = x.permute(0, 2, 1)
        
        # CNN Block with ReLU
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # DNN Block with Tanh/Softsign
        x = self.dense_activation(self.fc1(x))
        x = self.dropout(x)
        x = self.dense_activation(self.fc2(x))
        x = self.dense_activation(self.fc3(x))
        x = self.dense_activation(self.fc4(x))
        x = self.dense_activation(self.fc5(x))
        x = self.dense_activation(self.fc6(x))
        
        return x
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


print("Models loaded: AutoEncoder1D (FingerFlex), BC4D4")

In [ ]:
class ValidationCallback(Callback):
    """
    Callback for FingerFlex model - calculates correlation at end of each validation epoch
    """
    def __init__(self, val_x, val_y, fg_num):
        super().__init__()
        self.val_x = val_x.T
        self.val_y = val_y.T
        self.fg_num = fg_num

    def on_validation_epoch_end(self, trainer, pl_module):
        with torch.no_grad():
            SIZE = 64
            bound = self.val_x.shape[0]//SIZE *SIZE

            X_test = self.val_x[:bound]
            y_test = self.val_y[:bound]
            x_batch = torch.from_numpy(X_test).float().to("cuda:0")

            x_batch = x_batch.T
            x_batch = torch.unsqueeze(x_batch, 0)

            y_hat = pl_module.model(x_batch)[0]
            y_hat = y_hat.cpu().detach().numpy()
            STRIDE = 1
            y_prediction = y_hat.T[::int(STRIDE*(DOWNSAMPLE_FS/100)), :]
            y_prediction = scipy.ndimage.gaussian_filter1d(y_prediction.T,sigma=6).T

            y_test = y_test[::int(STRIDE*(DOWNSAMPLE_FS/100)), :]

            h, w = self.fg_num//2, self.fg_num - self.fg_num//2
            fig, ax = plt.subplots(h, w, figsize = (h*5, w*6), sharex=True, sharey=True)
            corrs = []

            for roi in range(self.fg_num):
                y_hat = y_prediction[:, roi]
                y_test_roi = y_test[:, roi]
                corr_tmp = corr_metric(y_hat, y_test_roi)
                corrs.append(corr_tmp)
                axi = ax.flat[roi]
                axi.plot(y_hat, label= 'prediction')
                axi.plot(y_test_roi, label = 'true')
                axi.set_title("RoI {}_corr {:.2f}".format(roi, corr_tmp))

            corr_mean = np.mean(corrs)
            pl_module.log("corr_mean_val", corr_mean, on_step=False, on_epoch=True, prog_bar=True, logger=True)
            wandb.log({f"plots": fig})
            plt.close(fig)


class BC4D4ValidationCallback(Callback):
    """
    Callback for BC4D4 model - calculates Pearson correlation at end of each validation epoch
    """
    def __init__(self, val_x, val_y, finger_idx=None):
        super().__init__()
        self.val_x = val_x.astype('float32')  # Shape: (time, electrodes, 1)
        self.val_y = val_y.astype('float32')  # Shape: (time, 5)
        self.finger_idx = finger_idx
        self.finger_names = ['Thumb', 'Index', 'Middle', 'Ring', 'Little']

    def on_validation_epoch_end(self, trainer, pl_module):
        with torch.no_grad():
            # Process in batches
            batch_size = 512
            n_samples = len(self.val_x)
            
            all_preds = []
            x_tensor = torch.from_numpy(self.val_x).float().to("cuda:0")
            
            for i in range(0, n_samples, batch_size):
                batch = x_tensor[i:i+batch_size]
                pred = pl_module.model(batch)
                all_preds.append(pred.cpu().numpy())
            
            y_pred = np.concatenate(all_preds, axis=0)
            
            # Get the appropriate ground truth
            if self.finger_idx is not None:
                y_true = self.val_y[:, self.finger_idx]
                finger_name = self.finger_names[self.finger_idx]
                
                # Compute correlation
                corr = corr_metric(y_true.flatten(), y_pred.flatten())
                
                pl_module.log(f"val_corr_{finger_name}", corr, on_step=False, on_epoch=True, prog_bar=True)
                pl_module.log("corr_mean_val", corr, on_step=False, on_epoch=True, prog_bar=True, logger=True)
                
                # Create plot
                fig, ax = plt.subplots(figsize=(12, 4))
                n_plot = min(1000, len(y_true))
                ax.plot(y_true[:n_plot], label='True', alpha=0.7)
                ax.plot(y_pred[:n_plot].flatten(), label='Predicted', alpha=0.7)
                ax.set_title(f'{finger_name} - Correlation: {corr:.4f}')
                ax.legend()
                ax.set_xlabel('Sample')
                ax.set_ylabel('Finger Position')
            else:
                # All 5 fingers
                corrs = []
                fig, axes = plt.subplots(1, 5, figsize=(20, 4))
                
                for i, name in enumerate(self.finger_names):
                    corr = corr_metric(self.val_y[:, i], y_pred[:, i])
                    corrs.append(corr)
                    pl_module.log(f"val_corr_{name}", corr, on_step=False, on_epoch=True)
                    
                    n_plot = min(500, len(self.val_y))
                    axes[i].plot(self.val_y[:n_plot, i], label='True', alpha=0.7)
                    axes[i].plot(y_pred[:n_plot, i], label='Pred', alpha=0.7)
                    axes[i].set_title(f'{name}: {corr:.3f}')
                
                corr_mean = np.mean(corrs)
                pl_module.log("corr_mean_val", corr_mean, on_step=False, on_epoch=True, prog_bar=True, logger=True)
                fig.suptitle(f'Average Correlation: {corr_mean:.4f}')
            
            plt.tight_layout()
            wandb.log({"val_predictions": fig})
            plt.close(fig)


class TestCallback:
    """
    Callback for FingerFlex test evaluation
    """
    def __init__(self, val_x, val_y, fg_num):
        super().__init__()
        self.val_x = val_x.T
        self.val_y = val_y.T
        self.fg_num = fg_num

    def test(self, pl_module):
        with torch.no_grad():
            SIZE = 64
            bound = self.val_x.shape[0]//SIZE *SIZE

            X_test = self.val_x[:bound]
            y_test = self.val_y[:bound]
            x_batch = torch.from_numpy(X_test).float().to("cuda:0")

            x_batch = x_batch.T
            x_batch = torch.unsqueeze(x_batch, 0)

            y_hat = pl_module.model(x_batch)[0]
            y_hat = y_hat.cpu().detach().numpy()
            STRIDE = 1
            y_prediction = y_hat.T[::int(STRIDE*(DOWNSAMPLE_FS/100)), :]
            y_prediction = scipy.ndimage.gaussian_filter1d(y_prediction.T,sigma=1).T

            y_test = y_test[::int(STRIDE*(DOWNSAMPLE_FS/100)), :]

            np.save(f"{pathlib.Path().resolve()}/res_npy/prediction2.npy", y_prediction)
            np.save(f"{pathlib.Path().resolve()}/res_npy/true2.npy", y_test)

            h, w = self.fg_num//2, self.fg_num - self.fg_num//2
            fig, ax = plt.subplots(h, w, figsize = (h*35, w*6), sharex=True, sharey=True)
            corrs = []

            for roi in range(self.fg_num):
                y_hat = y_prediction[:, roi]
                y_test_roi = y_test[:, roi]
                corr_tmp = corr_metric(y_hat, y_test_roi)
                corrs.append(corr_tmp)
                axi = ax.flat[roi]
                axi.plot(y_hat, label= 'prediction')
                axi.plot(y_test_roi, label = 'true')
                axi.set_title("RoI {}_corr {:.2f}".format(roi, corr_tmp))

            corr_mean = np.mean(corrs)
            plotly_fig = tls.mpl_to_plotly(fig)
            print(f"Mean correlation: {corr_mean}")
            plotly_fig.write_html("res.html")


class BC4D4TestCallback:
    """
    Test callback for BC4D4 model
    """
    def __init__(self, val_x, val_y, finger_idx=None):
        self.val_x = val_x.astype('float32')
        self.val_y = val_y.astype('float32')
        self.finger_idx = finger_idx
        self.finger_names = ['Thumb', 'Index', 'Middle', 'Ring', 'Little']

    def test(self, pl_module):
        with torch.no_grad():
            batch_size = 512
            n_samples = len(self.val_x)
            
            all_preds = []
            x_tensor = torch.from_numpy(self.val_x).float().to("cuda:0")
            
            for i in range(0, n_samples, batch_size):
                batch = x_tensor[i:i+batch_size]
                pred = pl_module.model(batch)
                all_preds.append(pred.cpu().numpy())
            
            y_pred = np.concatenate(all_preds, axis=0)
            
            print("=" * 60)
            print("BC4D4 TEST RESULTS")
            print("=" * 60)
            
            if self.finger_idx is not None:
                y_true = self.val_y[:, self.finger_idx]
                corr = corr_metric(y_true.flatten(), y_pred.flatten())
                print(f"{self.finger_names[self.finger_idx]}: {corr:.4f}")
            else:
                corrs = []
                for i, name in enumerate(self.finger_names):
                    corr = corr_metric(self.val_y[:, i], y_pred[:, i])
                    corrs.append(corr)
                    print(f"  {name}: {corr:.4f}")
                print(f"\n  Average: {np.mean(corrs):.4f}")
            
            print("=" * 60)

In [ ]:
# =============================================================================
# MODEL AND DATAMODULE INITIALIZATION
# =============================================================================

if MODEL_MODE == 'fingerflex':
    # FingerFlex Model Configuration
    SAMPLE_LEN = 256  # Window size
    finger_num = 5    # Number of fingers

    hp_autoencoder = dict(
        channels = [32, 32, 64, 64, 128, 128], 
        kernel_sizes=[7, 7, 5, 5, 5],
        strides=[2, 2, 2, 2, 2],
        dilation=[1, 1, 1, 1, 1],
        n_electrodes = CHANNELS_NUM,
        n_freqs = WAVELET_NUM,
        n_channels_out = finger_num
    )

    model = AutoEncoder1D(**hp_autoencoder).to("cuda:0")
    lightning_wrapper = BaseEcogFingerflexModel(model)
    
    dm = EcogFingerflexDatamodule(
        sample_len=SAMPLE_LEN, 
        add_name="",
        model_mode='fingerflex'
    )
    
    print("=" * 60)
    print("FINGERFLEX MODEL (U-Net)")
    print("=" * 60)
    summary(model, torch.zeros(4, CHANNELS_NUM, WAVELET_NUM, SAMPLE_LEN).to("cuda:0"), show_input=False)

elif MODEL_MODE == 'bc4d4':
    # BC4D4 Model Configuration
    # Paper trains ONE model per finger (output=1)
    n_outputs = 1 if BC4D4_FINGER_IDX is not None else 5
    finger_num = 1 if BC4D4_FINGER_IDX is not None else 5
    
    model = BC4D4(
        num_features=CHANNELS_NUM,
        activation=BC4D4_ACTIVATION,
        dropout_rate=0.1,
        n_outputs=n_outputs
    ).to("cuda:0")
    
    lightning_wrapper = BC4D4LightningModel(model, lr=1e-3)
    
    dm = EcogFingerflexDatamodule(
        sample_len=1,  # Not used for BC4D4
        add_name="_bc4d4",
        model_mode='bc4d4',
        finger_idx=BC4D4_FINGER_IDX,
        batch_size=64
    )
    
    print("=" * 60)
    print(f"BC4D4 MODEL (CNN+DNN) - {BC4D4_ACTIVATION.upper()} activation")
    print("=" * 60)
    print(f"Number of electrodes: {CHANNELS_NUM}")
    print(f"Number of outputs: {n_outputs}")
    if BC4D4_FINGER_IDX is not None:
        print(f"Training finger: {FINGER_NAMES[BC4D4_FINGER_IDX]} (paper methodology: per-finger models)")
    else:
        print("Training all 5 fingers at once")
    print(f"Total parameters: {model.count_parameters():,}")
    
    # Test forward pass
    test_input = torch.zeros(4, CHANNELS_NUM, 1).to("cuda:0")
    test_output = model(test_input)
    print(f"Input shape: {test_input.shape} -> Output shape: {test_output.shape}")

else:
    raise ValueError(f"Unknown MODEL_MODE: {MODEL_MODE}. Use 'fingerflex' or 'bc4d4'")

In [ ]:
SAVE_PATH = f"{pathlib.Path().resolve()}/data"

def load_data(ecog_data_path, fingerflex_data_path):
    ecog_data = np.load(ecog_data_path)
    fingerflex_data = np.load(fingerflex_data_path)
    return ecog_data, fingerflex_data

# Load validation data based on model mode
if MODEL_MODE == 'fingerflex':
    ecog_data_val, fingerflex_data_val = load_data(
        f"{SAVE_PATH}/val/ecog_data.npy", 
        f"{SAVE_PATH}/val/fingerflex_data.npy"
    )
elif MODEL_MODE == 'bc4d4':
    ecog_data_val, fingerflex_data_val = load_data(
        f"{SAVE_PATH}/val/ecog_data_bc4d4.npy", 
        f"{SAVE_PATH}/val/fingerflex_data_bc4d4.npy"
    )

print(f"Validation ECoG shape: {ecog_data_val.shape}")
print(f"Validation finger shape: {fingerflex_data_val.shape}")

In [ ]:
### TRAINING / TESTING ###
from pytorch_lightning.plugins.environments import LightningEnvironment

if TYPE == "train":
    wandb.init(project="BCI_comp")
    wandb_logger = WandbLogger()

    # Create checkpoint filename with finger name for BC4D4
    if MODEL_MODE == 'bc4d4' and BC4D4_FINGER_IDX is not None:
        ckpt_filename = f"bc4d4-{FINGER_NAMES[BC4D4_FINGER_IDX].lower()}-{{epoch:02d}}-{{corr_mean_val:.4f}}"
    else:
        ckpt_filename = f"{MODEL_MODE}-model-{{epoch:02d}}-{{corr_mean_val:.4f}}"

    checkpoint_callback = ModelCheckpoint(
        save_top_k=2,
        monitor="corr_mean_val",
        mode="max",
        dirpath="checkpoints",
        filename=ckpt_filename,
    )

    # Select appropriate validation callback based on model
    if MODEL_MODE == 'fingerflex':
        val_callback = ValidationCallback(ecog_data_val, fingerflex_data_val, finger_num)
    elif MODEL_MODE == 'bc4d4':
        val_callback = BC4D4ValidationCallback(ecog_data_val, fingerflex_data_val, BC4D4_FINGER_IDX)

    trainer = Trainer(
        accelerator='gpu', 
        devices=1,
        max_epochs=100 if MODEL_MODE == 'bc4d4' else 20,
        logger=wandb_logger, 
        plugins=[LightningEnvironment()],
        callbacks=[val_callback, checkpoint_callback]
    )

    print(f"\n{'='*60}")
    print(f"Starting training: {MODEL_MODE.upper()} model")
    if MODEL_MODE == 'bc4d4' and BC4D4_FINGER_IDX is not None:
        print(f"Finger: {FINGER_NAMES[BC4D4_FINGER_IDX]}")
    print(f"{'='*60}")
    
    trainer.fit(lightning_wrapper, dm)
    wandb.finish()

elif TYPE == "test":
    ### TEST MODE ###
    if MODEL_MODE == 'fingerflex':
        trained_model = BaseEcogFingerflexModel.load_from_checkpoint(
            checkpoint_path=model_to_test,
            model=AutoEncoder1D(**hp_autoencoder)
        )
        trained_model = trained_model.cuda()
        
        test_callback = TestCallback(ecog_data_val, fingerflex_data_val, finger_num)
        test_callback.test(trained_model)
        
    elif MODEL_MODE == 'bc4d4':
        # For BC4D4, you need to specify the checkpoint path
        bc4d4_model = BC4D4(
            num_features=CHANNELS_NUM,
            activation=BC4D4_ACTIVATION,
            n_outputs=1 if BC4D4_FINGER_IDX is not None else 5
        )
        
        # Load checkpoint if exists
        if BC4D4_FINGER_IDX is not None:
            bc4d4_checkpoint = f"{pathlib.Path().resolve()}/checkpoints/bc4d4-{FINGER_NAMES[BC4D4_FINGER_IDX].lower()}-best.ckpt"
        else:
            bc4d4_checkpoint = f"{pathlib.Path().resolve()}/checkpoints/bc4d4-model-best.ckpt"
            
        if pathlib.Path(bc4d4_checkpoint).exists():
            trained_model = BC4D4LightningModel.load_from_checkpoint(
                checkpoint_path=bc4d4_checkpoint,
                model=bc4d4_model
            )
            trained_model = trained_model.cuda()
            
            test_callback = BC4D4TestCallback(ecog_data_val, fingerflex_data_val, BC4D4_FINGER_IDX)
            test_callback.test(trained_model)
        else:
            print(f"BC4D4 checkpoint not found at: {bc4d4_checkpoint}")
            print("Please train the model first or update the checkpoint path.")